# 03 - Feature Engineering & Relational Merging

### Overview
- Merge relational datasets: store metadata, oil prices, holiday calendar, transactions.
- Build temporal calendar features (day, month, year, day-of-week, week-of-year, weekend flag).
- Interpolate missing oil prices and flag national holidays.
- Export engineered datasets: `train_features.csv` and `test_features.csv`.


In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [9]:
import os
base_dir = ".." if os.path.exists("../data") else "."

train = pd.read_csv(f"{base_dir}/data/raw/train.csv")
test = pd.read_csv(f"{base_dir}/data/raw/test.csv")

stores = pd.read_csv(f"{base_dir}/data/raw/stores.csv")
oil = pd.read_csv(f"{base_dir}/data/raw/oil.csv")
holidays = pd.read_csv(f"{base_dir}/data/raw/holidays_events.csv")
transactions = pd.read_csv(f"{base_dir}/data/raw/transactions.csv")


In [ ]:
for df in [train, test, oil, holidays, transactions]:
    df["date"] = pd.to_datetime(df["date"])

In [11]:
train.shape,test.shape

((3000888, 6), (28512, 5))

In [12]:
train.head()

,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0


In [14]:
def date_features(df):
    df=df.copy()
    df['year']=df['date'].dt.year
    df['month']=df['date'].dt.month
    df['day']=df['date'].dt.day
    df['week_name']=df['date'].dt.dayofweek
    df['year_name']=df['date'].dt.dayofyear
    df['week_of_year']=df['date'].dt.isocalendar().week.astype(int)
    df["quarter"] = df["date"].dt.quarter
    df["is_weekend"] = (df["week_name"].isin([5, 6]).astype(int))
    return df

train=date_features(train)
test=date_features(test)    
train.head()

,id,date,store_nbr,family,sales,onpromotion,year,month,day,week_name,year_name,week_of_year,quarter,is_weekend
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,2013,1,1,1,1,1,1,0
1,1,2013-01-01,1,BABY CARE,0.0,0,2013,1,1,1,1,1,1,0
2,2,2013-01-01,1,BEAUTY,0.0,0,2013,1,1,1,1,1,1,0
3,3,2013-01-01,1,BEVERAGES,0.0,0,2013,1,1,1,1,1,1,0
4,4,2013-01-01,1,BOOKS,0.0,0,2013,1,1,1,1,1,1,0


In [16]:
stores.head()

,store_nbr,city,state,type,cluster
0,1,Quito,Pichincha,D,13
1,2,Quito,Pichincha,D,13
2,3,Quito,Pichincha,D,8
3,4,Quito,Pichincha,D,9
4,5,Santo Domingo,Santo Domingo de los Tsachilas,D,4


In [17]:
train=train.merge(stores,on='store_nbr',how='left')
test=test.merge(stores,on='store_nbr',how='left')
train.head()

,id,date,store_nbr,family,sales,onpromotion,year,month,day,week_name,year_name,week_of_year,quarter,is_weekend,city,state,type,cluster
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,2013,1,1,1,1,1,1,0,Quito,Pichincha,D,13
1,1,2013-01-01,1,BABY CARE,0.0,0,2013,1,1,1,1,1,1,0,Quito,Pichincha,D,13
2,2,2013-01-01,1,BEAUTY,0.0,0,2013,1,1,1,1,1,1,0,Quito,Pichincha,D,13
3,3,2013-01-01,1,BEVERAGES,0.0,0,2013,1,1,1,1,1,1,0,Quito,Pichincha,D,13
4,4,2013-01-01,1,BOOKS,0.0,0,2013,1,1,1,1,1,1,0,Quito,Pichincha,D,13


In [18]:
oil.head()

,date,dcoilwtico
0,2013-01-01,NaN
1,2013-01-02,93.14
2,2013-01-03,92.97
3,2013-01-04,93.12
4,2013-01-07,93.20


In [20]:
oil=oil.sort_values('date').copy()
oil.isnull().sum()

date           0
dcoilwtico    43
dtype: int64

In [23]:
oil['dcoilwtico']=oil['dcoilwtico'].bfill()
oil.isnull().sum()

date          0
dcoilwtico    0
dtype: int64

In [25]:
train=train.merge(oil,how='left',on='date')
test=test.merge(oil,how='left',on='date')
train.head()

,id,date,store_nbr,family,sales,onpromotion,year,month,day,week_name,year_name,week_of_year,quarter,is_weekend,city,state,type,cluster,dcoilwtico
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,2013,1,1,1,1,1,1,0,Quito,Pichincha,D,13,93.14
1,1,2013-01-01,1,BABY CARE,0.0,0,2013,1,1,1,1,1,1,0,Quito,Pichincha,D,13,93.14
2,2,2013-01-01,1,BEAUTY,0.0,0,2013,1,1,1,1,1,1,0,Quito,Pichincha,D,13,93.14
3,3,2013-01-01,1,BEVERAGES,0.0,0,2013,1,1,1,1,1,1,0,Quito,Pichincha,D,13,93.14
4,4,2013-01-01,1,BOOKS,0.0,0,2013,1,1,1,1,1,1,0,Quito,Pichincha,D,13,93.14


In [26]:
train.isnull().sum()

id                   0
date                 0
store_nbr            0
family               0
sales                0
onpromotion          0
year                 0
month                0
day                  0
week_name            0
year_name            0
week_of_year         0
quarter              0
is_weekend           0
city                 0
state                0
type                 0
cluster              0
dcoilwtico      857142
dtype: int64

In [29]:
train['dcoilwtico']=train['dcoilwtico'].ffill()
test['dcoilwtico']=test['dcoilwtico'].ffill()
train.isnull().sum(),train.isnull().sum()

(id              0
 date            0
 store_nbr       0
 family          0
 sales           0
 onpromotion     0
 year            0
 month           0
 day             0
 week_name       0
 year_name       0
 week_of_year    0
 quarter         0
 is_weekend      0
 city            0
 state           0
 type            0
 cluster         0
 dcoilwtico      0
 dtype: int64,
 id              0
 date            0
 store_nbr       0
 family          0
 sales           0
 onpromotion     0
 year            0
 month           0
 day             0
 week_name       0
 year_name       0
 week_of_year    0
 quarter         0
 is_weekend      0
 city            0
 state           0
 type            0
 cluster         0
 dcoilwtico      0
 dtype: int64)

In [31]:
for df in [train, test]:

    df["onpromotion"] = df["onpromotion"].fillna(0)

    df["has_promotion"] = (
        df["onpromotion"] > 0
    ).astype(int)

    df["promotion_intensity"] = np.log1p(
        df["onpromotion"]
    )
train.head()

,id,date,store_nbr,family,sales,onpromotion,year,month,day,week_name,...,week_of_year,quarter,is_weekend,city,state,type,cluster,dcoilwtico,has_promotion,promotion_intensity
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,2013,1,1,1,...,1,1,0,Quito,Pichincha,D,13,93.14,0,0.0
1,1,2013-01-01,1,BABY CARE,0.0,0,2013,1,1,1,...,1,1,0,Quito,Pichincha,D,13,93.14,0,0.0
2,2,2013-01-01,1,BEAUTY,0.0,0,2013,1,1,1,...,1,1,0,Quito,Pichincha,D,13,93.14,0,0.0
3,3,2013-01-01,1,BEVERAGES,0.0,0,2013,1,1,1,...,1,1,0,Quito,Pichincha,D,13,93.14,0,0.0
4,4,2013-01-01,1,BOOKS,0.0,0,2013,1,1,1,...,1,1,0,Quito,Pichincha,D,13,93.14,0,0.0


In [32]:
national_holidays = holidays[
    holidays["locale"] == "National"
].copy()

national_holiday_features = (
    national_holidays.groupby("date")
    .agg(
        is_national_holiday=("type", lambda x: 1),
        holiday_count=("type", "count")
    )
    .reset_index()
)
train = train.merge(
    national_holiday_features,
    on="date",
    how="left"
)

test = test.merge(
    national_holiday_features,
    on="date",
    how="left"
)
for df in [train, test]:
    df["is_national_holiday"] = (
        df["is_national_holiday"].fillna(0).astype(int)
    )

    df["holiday_count"] = df["holiday_count"].fillna(0)

In [33]:
train.head()

,id,date,store_nbr,family,sales,onpromotion,year,month,day,week_name,...,is_weekend,city,state,type,cluster,dcoilwtico,has_promotion,promotion_intensity,is_national_holiday,holiday_count
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,2013,1,1,1,...,0,Quito,Pichincha,D,13,93.14,0,0.0,1,1.0
1,1,2013-01-01,1,BABY CARE,0.0,0,2013,1,1,1,...,0,Quito,Pichincha,D,13,93.14,0,0.0,1,1.0
2,2,2013-01-01,1,BEAUTY,0.0,0,2013,1,1,1,...,0,Quito,Pichincha,D,13,93.14,0,0.0,1,1.0
3,3,2013-01-01,1,BEVERAGES,0.0,0,2013,1,1,1,...,0,Quito,Pichincha,D,13,93.14,0,0.0,1,1.0
4,4,2013-01-01,1,BOOKS,0.0,0,2013,1,1,1,...,0,Quito,Pichincha,D,13,93.14,0,0.0,1,1.0


In [34]:
print("Training shape:", train.shape)
print("Testing shape:", test.shape)

print("\nFeatures:")
print(train.columns.tolist())

print("\nMissing values:")
print(train.isnull().sum().sort_values(ascending=False).head(15))

display(train.head())

Training shape: (3000888, 23)
Testing shape: (28512, 22)

Features:
['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion', 'year', 'month', 'day', 'week_name', 'year_name', 'week_of_year', 'quarter', 'is_weekend', 'city', 'state', 'type', 'cluster', 'dcoilwtico', 'has_promotion', 'promotion_intensity', 'is_national_holiday', 'holiday_count']

Missing values:
id              0
date            0
store_nbr       0
family          0
sales           0
onpromotion     0
year            0
month           0
day             0
week_name       0
year_name       0
week_of_year    0
quarter         0
is_weekend      0
city            0
dtype: int64


,id,date,store_nbr,family,sales,onpromotion,year,month,day,week_name,...,is_weekend,city,state,type,cluster,dcoilwtico,has_promotion,promotion_intensity,is_national_holiday,holiday_count
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,2013,1,1,1,...,0,Quito,Pichincha,D,13,93.14,0,0.0,1,1.0
1,1,2013-01-01,1,BABY CARE,0.0,0,2013,1,1,1,...,0,Quito,Pichincha,D,13,93.14,0,0.0,1,1.0
2,2,2013-01-01,1,BEAUTY,0.0,0,2013,1,1,1,...,0,Quito,Pichincha,D,13,93.14,0,0.0,1,1.0
3,3,2013-01-01,1,BEVERAGES,0.0,0,2013,1,1,1,...,0,Quito,Pichincha,D,13,93.14,0,0.0,1,1.0
4,4,2013-01-01,1,BOOKS,0.0,0,2013,1,1,1,...,0,Quito,Pichincha,D,13,93.14,0,0.0,1,1.0


In [35]:
os.makedirs(f"{base_dir}/data/processed", exist_ok=True)
train.to_csv(
    f"{base_dir}/data/processed/train_features.csv",
    index=False
)

test.to_csv(
    f"{base_dir}/data/processed/test_features.csv",
    index=False
)
